In [5]:
import os
import json
import zipfile

import pandas as pd

RB_MASTER_CSV = os.path.join("rb_analysis_master.csv")
ESPN_ZIP_PATH = os.path.join("espn_jsons.zip")

OUT_CSV = "final_rb_analysis_master.csv"

def build_word_to_abbrev_mapping(zip_path: str):
    """
    Read one ESPN json file and build mapping:
      last word of team name ('Chargers') -> abbrev ('LAC')
    """
    with zipfile.ZipFile(zip_path, "r") as z:
        json_files = [n for n in z.namelist() if n.endswith(".json")]
        if not json_files:
            raise FileNotFoundError("No .json files found inside espn_jsons.zip")
        sample = json_files[0]
        data = json.loads(z.read(sample).decode("utf-8"))

    word_to_abbrev = {}
    for team in data:
        name = team.get("name", "")
        abbr = team.get("abbrev", "")
        if not name or not abbr:
            continue
        last_word = name.split()[-1].lower()
        word_to_abbrev[last_word] = abbr
    return word_to_abbrev


def make_rb_team_normalizer(word_to_abbrev: dict):
    """
    Returns a function normalize_rb_team(team_str) that:
      - For 'ARI/NYJ' etc, takes only the first part ('ARI')
    """
    def normalize_rb_team(team: str):
        if not isinstance(team, str):
            return None

        # If multiple teams, take the first team (e.g. 'ARI' from 'ARI/NYJ')
        if "/" in team:
            team = team.split("/")[0]

        t = team.strip()

        # If already an abbrev like 'ARI', 'NE', 'NYJ', etc.
        if t.isupper():
            return t

        last = t.split()[-1].lower()

        if last in word_to_abbrev:
            return word_to_abbrev[last]

        if t.lower() in word_to_abbrev:
            return word_to_abbrev[t.lower()]

        return None

    return normalize_rb_team


# ===============================
# 2. Build ESPN team/year stats
# ===============================

word_to_abbrev = build_word_to_abbrev_mapping(ESPN_ZIP_PATH)
normalize_rb_team = make_rb_team_normalizer(word_to_abbrev)

espn_rows = []

with zipfile.ZipFile(ESPN_ZIP_PATH, "r") as z:
    for fname in z.namelist():
        if not fname.endswith(".json"):
            continue

        season_data = json.loads(z.read(fname).decode("utf-8"))
        for team in season_data:
            season = team.get("season")
            abbrev = team.get("abbrev")
            record = team.get("record", {})
            stats = team.get("stats", {})

            wins = record.get("wins")
            losses = record.get("losses")
            rush_yards = stats.get("offense_rushing_yards")
            pass_yards = stats.get("offense_passing_yards")

            espn_rows.append(
                {
                    "Year": season,
                    "team_abbrev": abbrev,
                    "Wins": wins,
                    "Losses": losses,
                    "Team_RushYds": rush_yards,
                    "Team_PassYds": pass_yards,
                }
            )

df_espn = pd.DataFrame(espn_rows)

# Win percentage
df_espn["WinPct"] = df_espn["Wins"] / (df_espn["Wins"] + df_espn["Losses"])

# ===============================
# 3. Load RB master and add team_abbrev
# ===============================

df_rb = pd.read_csv(RB_MASTER_CSV)

# Create team_abbrev column based on RB 'Team' field
df_rb["team_abbrev"] = df_rb["Team"].apply(normalize_rb_team)

# ===============================
# 4. Merge on (Year, team_abbrev)
# ===============================

df_merged = df_rb.merge(
    df_espn,
    how="left",
    on=["Year", "team_abbrev"],
    validate="m:1",
)

# ===============================
# 5. Save
# ===============================

df_merged.to_csv(OUT_CSV, index=False)

print("Merged shape:", df_merged.shape)
print("Saved to:", OUT_CSV)
df_merged.head()


Merged shape: (5019, 34)
Saved to: final_rb_analysis_master.csv


,Player,Team,Age,G,GS,rAtt,rYds,rTD,r1D,rLng,...,Unnamed: 10,inflated_value,inflated_apy,inflated_guaranteed,team_abbrev,Wins,Losses,Team_RushYds,Team_PassYds,WinPct
0,LaDainian Tomlinson,Chargers,22,16,16,339,1236,10,68,54,...,NaN,166310094.0,27718349.0,57307687.0,LAC,3.0,9.0,1695.0,3685.0,0.25
1,LaDainian Tomlinson,Chargers,22,16,16,339,1236,10,68,54,...,NaN,40574797.0,13524931.0,17535122.0,LAC,3.0,9.0,1695.0,3685.0,0.25
2,LaDainian Tomlinson,Chargers,22,16,16,339,1236,10,68,54,...,NaN,65238484.0,10873081.0,47634448.0,LAC,3.0,9.0,1695.0,3685.0,0.25
3,LaDainian Tomlinson,Chargers,22,16,16,339,1236,10,68,54,...,NaN,11146357.0,5573178.0,5410853.0,LAC,3.0,9.0,1695.0,3685.0,0.25
4,LaDainian Tomlinson,Chargers,22,16,16,339,1236,10,68,54,...,NaN,3073229.0,3073229.0,2609346.0,LAC,3.0,9.0,1695.0,3685.0,0.25
